# 02. Querying the Graph

Coming off [NB1](./01_introduction.ipynb): you've seen `MATCH`, you've drawn one two-hop pattern. That's the happy path.

**This is the walk.** Same shape of work, more moves. We drop the three-node toy and load the full movie graph (30 people, 11 movies, 8 studios, 54 edges) from `scripts/build_corpus.py`. Then we work through the Cypher you actually write for graph RAG: filtered MATCH, multi-hop traversals, aggregations, OPTIONAL MATCH, parameterized queries.

Still no LLM. The point is to get fluent in the query language before NB3 has an LLM try to write it for you.

## Load the movie graph

`scripts/build_corpus.py` writes the seed graph to `data/movies.kuzu` (a single file). The `build_if_missing()` call is idempotent: it does nothing if the file's already there. Safe to re-run.

If you want to start over, delete the file or pass `--force` to the script.

In [1]:
from helpers import get_kuzu_conn
from scripts.build_corpus import build_if_missing

build_if_missing()
conn = get_kuzu_conn()

print("Tables in the database:")
conn.execute("CALL show_tables() RETURN *").get_as_df()

Tables in the database:


,id,name,type,database name,comment
0,0,Person,NODE,local(kuzu),
1,1,Movie,NODE,local(kuzu),
2,6,DIRECTED,REL,local(kuzu),
3,2,Studio,NODE,local(kuzu),
4,8,ACTED_IN,REL,local(kuzu),
5,3,Script,NODE,local(kuzu),
6,4,Scene,NODE,local(kuzu),
7,10,PRODUCED_BY,REL,local(kuzu),
8,12,HAS_SCRIPT,REL,local(kuzu),
9,14,HAS_SCENE,REL,local(kuzu),


## How big is the graph?

A quick `count(n)` per node label. `label(n)` returns the node's table name, which lets us group without writing five separate queries.

In [2]:
conn.execute("""
    MATCH (n)
    RETURN label(n) AS kind, count(n) AS n
    ORDER BY n DESC
""").get_as_df()

,kind,n
0,Scene,1809
1,Person,30
2,Movie,11
3,Script,11
4,Studio,8


## Filtering with WHERE

`MATCH` finds the pattern, `WHERE` narrows it down. Two examples, both numeric: people born after a year, films before a year. `WHERE` also handles string equality (`name = '...'`), `STARTS WITH`, `CONTAINS`, list membership with `IN`, and the rest of the usual suspects.

In [3]:
conn.execute("""
    MATCH (p:Person)
    WHERE p.born_year > 1970
    RETURN p.name AS name, p.born_year AS born
    ORDER BY born
""").get_as_df()

,name,born
0,Leonardo DiCaprio,1974
1,Christian Bale,1974
2,Marion Cotillard,1975
3,Kate Winslet,1975
4,Jordan Peele,1979
5,Joseph Gordon-Levitt,1981
6,Scarlett Johansson,1984
7,Allison Williams,1988
8,Daniel Kaluuya,1989


In [4]:
conn.execute("""
    MATCH (m:Movie)
    WHERE m.year < 2000
    RETURN m.title AS title, m.year AS year
    ORDER BY year
""").get_as_df()

,title,year
0,Reservoir Dogs,1992
1,Pulp Fiction,1994
2,The Matrix,1999


## Two-hop traversal: co-actors

The pattern from NB1, with the `WHERE` we promised to add. `<>` is "not equal". This excludes Keanu from his own co-actor list.

In [5]:
conn.execute("""
    MATCH (keanu:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)<-[:ACTED_IN]-(other:Person)
    WHERE other.name <> 'Keanu Reeves'
    RETURN DISTINCT other.name AS co_actor, m.title AS in_movie
    ORDER BY co_actor, in_movie
""").get_as_df()

,co_actor,in_movie
0,Carrie-Anne Moss,The Matrix
1,Carrie-Anne Moss,The Matrix Reloaded
2,Joe Pantoliano,The Matrix
3,Laurence Fishburne,The Matrix
4,Laurence Fishburne,The Matrix Reloaded
5,Willem Dafoe,John Wick


## Three-hop traversal: directors you've worked with

Same shape, longer path. "Which directors has Keanu Reeves worked under?"

The pattern is `(keanu)-[:ACTED_IN]->(film)<-[:DIRECTED]-(director)`. Three nodes, two relationships, one query.

In [6]:
conn.execute("""
    MATCH (keanu:Person {name: 'Keanu Reeves'})-[:ACTED_IN]->(m:Movie)<-[:DIRECTED]-(d:Person)
    RETURN DISTINCT d.name AS director, m.title AS via_film
    ORDER BY director
""").get_as_df()

,director,via_film
0,Chad Stahelski,John Wick
1,The Wachowskis,The Matrix Reloaded
2,The Wachowskis,The Matrix


## Aggregations: count, collect

Cypher's `count()` counts matching rows. `collect()` rolls a column up into a list. Together they're how you build a profile of a node without writing N queries.

How many films has each director made?

In [7]:
conn.execute("""
    MATCH (p:Person)-[:DIRECTED]->(m:Movie)
    RETURN p.name AS director, count(m) AS films
    ORDER BY films DESC, director
""").get_as_df()

,director,films
0,Christopher Nolan,3
1,Quentin Tarantino,3
2,The Wachowskis,2
3,Chad Stahelski,1
4,Jordan Peele,1
5,Michel Gondry,1


And who acts in each director's films?

In [8]:
conn.execute("""
    MATCH (d:Person)-[:DIRECTED]->(m:Movie)<-[:ACTED_IN]-(a:Person)
    RETURN d.name AS director, collect(DISTINCT a.name) AS actors
    ORDER BY director
""").get_as_df()

,director,actors
0,Chad Stahelski,"[Keanu Reeves, Willem Dafoe]"
1,Christopher Nolan,"[Guy Pearce, Carrie-Anne Moss, Joe Pantoliano,..."
2,Jordan Peele,"[Daniel Kaluuya, Allison Williams]"
3,Michel Gondry,"[Jim Carrey, Kate Winslet]"
4,Quentin Tarantino,"[John Travolta, Samuel L. Jackson, Uma Thurman..."
5,The Wachowskis,"[Keanu Reeves, Carrie-Anne Moss, Laurence Fish..."


## OPTIONAL MATCH

A regular `MATCH` drops a row if any part of the pattern fails to bind. `OPTIONAL MATCH` keeps the row, with `NULL` where the optional part didn't match. The graph SQL equivalent of a `LEFT JOIN`.

Here every movie has a studio, so the result happens to have no nulls. The reason to use `OPTIONAL MATCH` is for partial patterns where some movies might not have a recorded studio: regular `MATCH` would silently drop them, `OPTIONAL MATCH` shows them with `NULL`.

In [9]:
conn.execute("""
    MATCH (d:Person)-[:DIRECTED]->(m:Movie)
    OPTIONAL MATCH (m)-[:PRODUCED_BY]->(s:Studio)
    RETURN d.name AS director, m.title AS movie, s.name AS studio
    ORDER BY director, movie
""").get_as_df()

,director,movie,studio
0,Chad Stahelski,John Wick,Lionsgate
1,Christopher Nolan,Inception,Warner Bros
2,Christopher Nolan,Memento,Newmarket
3,Christopher Nolan,The Prestige,Touchstone
4,Jordan Peele,Get Out,Universal
5,Michel Gondry,Eternal Sunshine of the Spotless Mind,Focus Features
6,Quentin Tarantino,Django Unchained,Weinstein
7,Quentin Tarantino,Pulp Fiction,Miramax
8,Quentin Tarantino,Reservoir Dogs,Miramax
9,The Wachowskis,The Matrix,Warner Bros


## Parameterized queries

Don't string-format user input into Cypher. Use `$placeholders` and a params dict. Same lesson as SQL: safer, faster (the query plan caches), and lets you reuse a query across calls.

`conn.execute(query, params)` takes a dict as the second argument.

In [10]:
conn.execute("""
    MATCH (p:Person {name: $name})-[:DIRECTED]->(m:Movie)
    RETURN m.title AS title, m.year AS year
    ORDER BY year
""", {"name": "Christopher Nolan"}).get_as_df()

,title,year
0,Memento,2000
1,The Prestige,2006
2,Inception,2010


## Two Kuzu-flavoured gotchas

While you're here, two small surprises that bit me while writing this notebook:

1. **`ORDER BY` references the projection alias, not the pattern variable.** If you `RETURN p.name AS director`, then `ORDER BY director` works. `ORDER BY p` doesn't (the error message is misleadingly "Variable not in scope"). Always order by what you returned.
2. **`cast` is a reserved word.** Use `actors` or another name as the alias for a `collect()` of actors. Kuzu reserves it because it parses `CAST(x AS TYPE)` for type coercion.

Both of these are minor papercuts that you'll learn once and never hit again. Worth flagging so you don't waste twenty minutes on either.

## Recap

| Pattern | Cypher | Use when |
|---|---|---|
| Filter | `MATCH ... WHERE x.y > 5` | Narrow the rows |
| Two-hop | `(a)-[:R]->(b)<-[:R]-(c)` | "Who's connected through this thing" |
| Three-hop | `(a)-[:R1]->(b)<-[:R2]-(c)` | "Who's two relationships away" |
| Count | `RETURN x, count(y)` | Group + tally |
| Collect | `RETURN x, collect(y)` | Group + list |
| OPTIONAL MATCH | `OPTIONAL MATCH (...)` | Like LEFT JOIN, keep rows with nulls |
| Parameters | `WHERE x = $param` + dict | Reusable, safe |

[NB3](./03_graph_rag_with_langchain.ipynb) wires an LLM in. We build the natural-language-to-Cypher loop two ways: by hand first, so we can see what's happening, then with LangChain's `KuzuQAChain` so we can see what it wraps.